<a href="https://colab.research.google.com/github/TanviUttla06/MLOps-Tanvi-B23BB1044/blob/Assignment-1/Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torch.cuda.amp import autocast, GradScaler
import time
import numpy as np

# --- CONFIGURATION ---
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. DATASET PREPARATION (The 70-10-20 Split) ---
def get_dataloaders(dataset_name='MNIST', batch_size=16):
    # ResNet requires some resizing/normalization.
    # We resize to 32x32 to be compatible with ResNet's downsampling
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # 1. Download/Load Data
    if dataset_name == 'MNIST':
        train_full = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        train_full = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    # 2. Merge and Re-split for strict 70-10-20 rule
    # Total MNIST/FashionMNIST size is 70,000
    full_dataset = ConcatDataset([train_full, test_full])
    total_size = len(full_dataset)

    train_size = int(0.70 * total_size)  # 49,000
    val_size = int(0.10 * total_size)    # 7,000
    test_size = total_size - train_size - val_size # 14,000 (20%)

    train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

    # 3. Create Loaders
    # pin_memory=True is usually faster on GPU
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True)

    return train_loader, val_loader, test_loader

# --- 2. MODEL SETUP (ResNet Modified for 1-Channel) ---
def get_model(model_name='resnet18'):
    if model_name == 'resnet18':
        model = torchvision.models.resnet18(pretrained=False)
    elif model_name == 'resnet50':
        model = torchvision.models.resnet50(pretrained=False)

    # MODIFY FIRST LAYER: Inputs are 1-channel (Grayscale), ResNet expects 3 (RGB)
    # We change input channels from 3 to 1.
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    return model.to(device)

# --- 3. TRAINING LOOP (With AMP) ---
def train_and_evaluate(config):
    # Unpack config
    ds_name = config['dataset']
    model_name = config['model']
    batch_size = config['batch_size']
    opt_name = config['optimizer']
    lr = config['lr']
    epochs = config['epochs']

    print(f"\n--- Running: {ds_name} | {model_name} | {opt_name} | Batch {batch_size} | LR {lr} ---")

    # Get Data
    train_loader, val_loader, test_loader = get_dataloaders(ds_name, batch_size)

    # Get Model
    model = get_model(model_name)

    # Optimizer
    if opt_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)

    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler() # For Automatic Mixed Precision (AMP)

    # Training Loop
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # AMP Forward Pass
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            # AMP Backward Pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        # Validation Loop (Optional: Add print if you want to track progress)
        # We skip printing every epoch to save space, but you can add it back.

    # FINAL TEST EVALUATION
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Result >> Test Accuracy: {acc:.2f}%")
    return acc

# --- 4. EXPERIMENT RUNNER ---
# This matches the table in your assignment image
# You can add FashionMNIST to this list to do Q1(a) fully.
# --- 4. EXPERIMENT RUNNER (MODIFIED FOR BOTH DATASETS) ---
experiments = [
    # Format: (Batch, Optim, LR, Model)
    (16, 'SGD', 0.001, 'resnet18'),
    (16, 'SGD', 0.0001, 'resnet18'),
    (16, 'Adam', 0.001, 'resnet18'),
    (16, 'Adam', 0.0001, 'resnet18'),

    (16, 'SGD', 0.001, 'resnet50'),
    (16, 'SGD', 0.0001, 'resnet50'),
    (16, 'Adam', 0.001, 'resnet50'),
    (16, 'Adam', 0.0001, 'resnet50'),

    # NOTE: You can add the Batch 32 experiments here if you need them later
]

datasets_to_run = ['MNIST', 'FashionMNIST']

print(f"Total Configurations to run: {len(datasets_to_run) * len(experiments)}")

for ds_name in datasets_to_run:
    print(f"\n{'='*20} STARTING {ds_name} EXPERIMENTS {'='*20}")

    for exp in experiments:
        batch, opt, lr, model = exp
        config = {
            'dataset': ds_name,   # Loops through MNIST then FashionMNIST
            'batch_size': batch,
            'optimizer': opt,
            'lr': lr,
            'model': model,
            'epochs': 5  # Keep this low (e.g. 5) to finish before the deadline!
        }

        # Run the training
        train_and_evaluate(config)

print("\nALL EXPERIMENTS COMPLETED.")

Using device: cuda
Total Configurations to run: 16

==================== STARTING MNIST EXPERIMENTS ====================

--- Running: MNIST | resnet18 | SGD | Batch 16 | LR 0.001 ---


/tmp/ipython-input-1286672463.py:91: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() # For Automatic Mixed Precision (AMP)
/tmp/ipython-input-1286672463.py:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Result >> Test Accuracy: 99.10%

--- Running: MNIST | resnet18 | SGD | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 98.56%

--- Running: MNIST | resnet18 | Adam | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 98.67%

--- Running: MNIST | resnet18 | Adam | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 98.71%

--- Running: MNIST | resnet50 | SGD | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 98.68%

--- Running: MNIST | resnet50 | SGD | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 97.74%

--- Running: MNIST | resnet50 | Adam | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 98.60%

--- Running: MNIST | resnet50 | Adam | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 97.96%

==================== STARTING FashionMNIST EXPERIMENTS ====================

--- Running: FashionMNIST | resnet18 | SGD | Batch 16 | LR 0.001 ---


100%|██████████| 26.4M/26.4M [00:02<00:00, 11.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 213kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.96MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 29.5MB/s]


Result >> Test Accuracy: 91.02%

--- Running: FashionMNIST | resnet18 | SGD | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 88.29%

--- Running: FashionMNIST | resnet18 | Adam | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 90.72%

--- Running: FashionMNIST | resnet18 | Adam | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 89.94%

--- Running: FashionMNIST | resnet50 | SGD | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 88.81%

--- Running: FashionMNIST | resnet50 | SGD | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 84.85%

--- Running: FashionMNIST | resnet50 | Adam | Batch 16 | LR 0.001 ---
Result >> Test Accuracy: 87.96%

--- Running: FashionMNIST | resnet50 | Adam | Batch 16 | LR 0.0001 ---
Result >> Test Accuracy: 88.23%

ALL EXPERIMENTS COMPLETED.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torch.cuda.amp import autocast, GradScaler
import time
import numpy as np

# --- CONFIGURATION ---
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. DATASET PREPARATION (The 70-10-20 Split) ---
def get_dataloaders(dataset_name='MNIST', batch_size=16):
    # ResNet requires some resizing/normalization.
    # We resize to 32x32 to be compatible with ResNet's downsampling
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # 1. Download/Load Data
    if dataset_name == 'MNIST':
        train_full = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        train_full = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    # 2. Merge and Re-split for strict 70-10-20 rule
    # Total MNIST/FashionMNIST size is 70,000
    full_dataset = ConcatDataset([train_full, test_full])
    total_size = len(full_dataset)

    train_size = int(0.70 * total_size)  # 49,000
    val_size = int(0.10 * total_size)    # 7,000
    test_size = total_size - train_size - val_size # 14,000 (20%)

    train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

    # 3. Create Loaders
    # pin_memory=True is usually faster on GPU
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True)

    return train_loader, val_loader, test_loader

# --- 2. MODEL SETUP (ResNet Modified for 1-Channel) ---
def get_model(model_name='resnet18'):
    if model_name == 'resnet18':
        model = torchvision.models.resnet18(pretrained=False)
    elif model_name == 'resnet50':
        model = torchvision.models.resnet50(pretrained=False)

    # MODIFY FIRST LAYER: Inputs are 1-channel (Grayscale), ResNet expects 3 (RGB)
    # We change input channels from 3 to 1.
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    return model.to(device)

# --- 3. TRAINING LOOP (With AMP) ---
def train_and_evaluate(config):
    # Unpack config
    ds_name = config['dataset']
    model_name = config['model']
    batch_size = config['batch_size']
    opt_name = config['optimizer']
    lr = config['lr']
    epochs = config['epochs']

    print(f"\n--- Running: {ds_name} | {model_name} | {opt_name} | Batch {batch_size} | LR {lr} ---")

    # Get Data
    train_loader, val_loader, test_loader = get_dataloaders(ds_name, batch_size)

    # Get Model
    model = get_model(model_name)

    # Optimizer
    if opt_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)

    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler() # For Automatic Mixed Precision (AMP)

    # Training Loop
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # AMP Forward Pass
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            # AMP Backward Pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        # Validation Loop (Optional: Add print if you want to track progress)
        # We skip printing every epoch to save space, but you can add it back.

    # FINAL TEST EVALUATION
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Result >> Test Accuracy: {acc:.2f}%")
    return acc

# --- 4. EXPERIMENT RUNNER ---
# This matches the table in your assignment image
# You can add FashionMNIST to this list to do Q1(a) fully.
# --- 4. EXPERIMENT RUNNER (MODIFIED FOR BOTH DATASETS) ---
experiments = [
    # Format: (Batch, Optim, LR, Model)
    (32, 'SGD', 0.001, 'resnet18'),
    (32, 'Adam', 0.0001, 'resnet18'),

    (32, 'SGD', 0.001, 'resnet50'),
    (32, 'Adam', 0.0001, 'resnet50'),

    # NOTE: You can add the Batch 32 experiments here if you need them later
]

datasets_to_run = ['MNIST', 'FashionMNIST']

print(f"Total Configurations to run: {len(datasets_to_run) * len(experiments)}")

for ds_name in datasets_to_run:
    print(f"\n{'='*20} STARTING {ds_name} EXPERIMENTS {'='*20}")

    for exp in experiments:
        batch, opt, lr, model = exp
        config = {
            'dataset': ds_name,   # Loops through MNIST then FashionMNIST
            'batch_size': batch,
            'optimizer': opt,
            'lr': lr,
            'model': model,
            'epochs': 5  # Keep this low (e.g. 5) to finish before the deadline!
        }

        # Run the training
        train_and_evaluate(config)

print("\nALL EXPERIMENTS COMPLETED.")

Using device: cuda
Total Configurations to run: 8

==================== STARTING MNIST EXPERIMENTS ====================

--- Running: MNIST | resnet18 | SGD | Batch 32 | LR 0.001 ---


/tmp/ipython-input-898904623.py:91: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() # For Automatic Mixed Precision (AMP)
/tmp/ipython-input-898904623.py:104: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Result >> Test Accuracy: 98.96%

--- Running: MNIST | resnet18 | Adam | Batch 32 | LR 0.0001 ---
Result >> Test Accuracy: 98.74%

--- Running: MNIST | resnet50 | SGD | Batch 32 | LR 0.001 ---
Result >> Test Accuracy: 98.72%

--- Running: MNIST | resnet50 | Adam | Batch 32 | LR 0.0001 ---
Result >> Test Accuracy: 97.03%

==================== STARTING FashionMNIST EXPERIMENTS ====================

--- Running: FashionMNIST | resnet18 | SGD | Batch 32 | LR 0.001 ---
Result >> Test Accuracy: 90.19%

--- Running: FashionMNIST | resnet18 | Adam | Batch 32 | LR 0.0001 ---
Result >> Test Accuracy: 89.83%

--- Running: FashionMNIST | resnet50 | SGD | Batch 32 | LR 0.001 ---
Result >> Test Accuracy: 88.61%

--- Running: FashionMNIST | resnet50 | Adam | Batch 32 | LR 0.0001 ---
Result >> Test Accuracy: 87.14%

ALL EXPERIMENTS COMPLETED.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torch.cuda.amp import autocast, GradScaler
import time
import numpy as np

# --- CONFIGURATION ---
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. DATASET PREPARATION (Updated to accept pin_memory) ---
def get_dataloaders(dataset_name='MNIST', batch_size=16, pin_memory=True):
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    if dataset_name == 'MNIST':
        train_full = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        train_full = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    full_dataset = ConcatDataset([train_full, test_full])
    total_size = len(full_dataset)
    train_size = int(0.70 * total_size)
    val_size = int(0.10 * total_size)
    test_size = total_size - train_size - val_size

    train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

    # PASS THE FLAG HERE
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=pin_memory)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=pin_memory)

    return train_loader, val_loader, test_loader

# --- 2. MODEL SETUP (ResNet Modified for 1-Channel) ---
def get_model(model_name='resnet18'):
    if model_name == 'resnet18':
        model = torchvision.models.resnet18(pretrained=False)
    elif model_name == 'resnet50':
        model = torchvision.models.resnet50(pretrained=False)

    # MODIFY FIRST LAYER: Inputs are 1-channel (Grayscale), ResNet expects 3 (RGB)
    # We change input channels from 3 to 1.
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    return model.to(device)

# # --- 3. TRAINING LOOP (Updated with Timer & Pin Memory support) ---
def train_and_evaluate(config):
    ds_name = config['dataset']
    model_name = config['model']
    batch_size = config['batch_size']
    opt_name = config['optimizer']
    lr = config['lr']
    epochs = config['epochs']
    # READ PIN_MEMORY (Default to True if not specified)
    pin_mem = config.get('pin_memory', True)

    print(f"\n--- Run: {ds_name} | {model_name} | {opt_name} | LR {lr} | PinMem: {pin_mem} ---")

    # Pass pin_mem to dataloaders
    train_loader, val_loader, test_loader = get_dataloaders(ds_name, batch_size, pin_mem)

    model = get_model(model_name)

    if opt_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)

    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler()

    # START TIMER
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    # END TIMER
    total_time = time.time() - start_time

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Result >> Accuracy: {acc:.2f}% | Time: {total_time:.2f}s")
    return acc

# --- 4. EXPERIMENT RUNNER ---
# This matches the table in your assignment image
# You can add FashionMNIST to this list to do Q1(a) fully.
# --- 4. EXPERIMENT RUNNER (MODIFIED FOR BOTH DATASETS) ---
experiments = [
    # Format: (Batch, Optim, LR, Model)
    (16, 'SGD', 0.001, 'resnet18'),
    (16, 'SGD', 0.0001, 'resnet18'),
    (16, 'Adam', 0.001, 'resnet18'),
    (16, 'Adam', 0.0001, 'resnet18'),

    (16, 'SGD', 0.001, 'resnet50'),
    (16, 'SGD', 0.0001, 'resnet50'),
    (16, 'Adam', 0.001, 'resnet50'),
    (16, 'Adam', 0.0001, 'resnet50'),

    (32, 'SGD', 0.001, 'resnet18'),
    (32, 'Adam', 0.0001, 'resnet18'),

    (32, 'SGD', 0.001, 'resnet50'),
    (32, 'Adam', 0.0001, 'resnet50'),



    # NOTE: You can add the Batch 32 experiments here if you need them later
]

datasets_to_run = ['MNIST', 'FashionMNIST']

print(f"Total Configurations to run: {len(datasets_to_run) * len(experiments)}")

for ds_name in datasets_to_run:
    print(f"\n{'='*20} STARTING {ds_name} EXPERIMENTS {'='*20}")

    for exp in experiments:
        batch, opt, lr, model = exp
        config = {
            'dataset': ds_name,   # Loops through MNIST then FashionMNIST
            'batch_size': batch,
            'optimizer': opt,
            'lr': lr,
            'model': model,
            'epochs': 2
        }

        # Run the training
        train_and_evaluate(config)

print("\nALL EXPERIMENTS COMPLETED.")

# --- PIN MEMORY COMPARISON STUDY ---
print("\n" + "="*30)
print("STARTING PIN_MEMORY COMPARISON (True vs False)")
print("="*30)

# We use one representative config to test the speed difference
base_config = {
    'dataset': 'MNIST',
    'batch_size': 16,
    'optimizer': 'Adam',
    'lr': 0.001,
    'model': 'resnet18',
    'epochs': 2 # Keeps epoch constant
}

# 1. Run with Pin Memory = TRUE
config_true = base_config.copy()
config_true['pin_memory'] = True
train_and_evaluate(config_true)

# 2. Run with Pin Memory = FALSE
config_false = base_config.copy()
config_false['pin_memory'] = False
train_and_evaluate(config_false)

Using device: cuda
Total Configurations to run: 24

==================== STARTING MNIST EXPERIMENTS ====================

--- Run: MNIST | resnet18 | SGD | LR 0.001 | PinMem: True ---


/tmp/ipython-input-867338103.py:83: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipython-input-867338103.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Result >> Accuracy: 98.66% | Time: 92.88s

--- Run: MNIST | resnet18 | SGD | LR 0.0001 | PinMem: True ---
Result >> Accuracy: 97.96% | Time: 91.83s

--- Run: MNIST | resnet18 | Adam | LR 0.001 | PinMem: True ---
Result >> Accuracy: 98.59% | Time: 95.34s

--- Run: MNIST | resnet18 | Adam | LR 0.0001 | PinMem: True ---
Result >> Accuracy: 98.74% | Time: 95.76s

--- Run: MNIST | resnet50 | SGD | LR 0.001 | PinMem: True ---
Result >> Accuracy: 98.13% | Time: 178.85s

--- Run: MNIST | resnet50 | SGD | LR 0.0001 | PinMem: True ---
Result >> Accuracy: 96.39% | Time: 181.82s

--- Run: MNIST | resnet50 | Adam | LR 0.001 | PinMem: True ---
Result >> Accuracy: 97.27% | Time: 201.85s

--- Run: MNIST | resnet50 | Adam | LR 0.0001 | PinMem: True ---
Result >> Accuracy: 96.79% | Time: 202.34s

--- Run: MNIST | resnet18 | SGD | LR 0.001 | PinMem: True ---
Result >> Accuracy: 98.54% | Time: 58.20s

--- Run: MNIST | resnet18 | Adam | LR 0.0001 | PinMem: True ---
Result >> Accuracy: 98.28% | Time: 59.76s

97.78571428571429

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, ConcatDataset
from torch.cuda.amp import autocast, GradScaler
import time
import numpy as np

# --- CONFIGURATION ---
# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 1. DATASET PREPARATION (The 70-10-20 Split) ---
def get_dataloaders(dataset_name='MNIST', batch_size=16):
    # ResNet requires some resizing/normalization.
    # We resize to 32x32 to be compatible with ResNet's downsampling
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # 1. Download/Load Data
    if dataset_name == 'MNIST':
        train_full = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        train_full = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_full = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    # 2. Merge and Re-split for strict 70-10-20 rule
    # Total MNIST/FashionMNIST size is 70,000
    full_dataset = ConcatDataset([train_full, test_full])
    total_size = len(full_dataset)

    train_size = int(0.70 * total_size)  # 49,000
    val_size = int(0.10 * total_size)    # 7,000
    test_size = total_size - train_size - val_size # 14,000 (20%)

    train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size])

    # 3. Create Loaders
    # pin_memory=False
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=False)

    return train_loader, val_loader, test_loader

# --- 2. MODEL SETUP (ResNet Modified for 1-Channel) ---
def get_model(model_name='resnet18'):
    if model_name == 'resnet18':
        model = torchvision.models.resnet18(pretrained=False)
    elif model_name == 'resnet50':
        model = torchvision.models.resnet50(pretrained=False)

    # MODIFY FIRST LAYER: Inputs are 1-channel (Grayscale), ResNet expects 3 (RGB)
    # We change input channels from 3 to 1.
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

    return model.to(device)

# --- 3. TRAINING LOOP (With AMP) ---
def train_and_evaluate(config):
    # Unpack config
    ds_name = config['dataset']
    model_name = config['model']
    batch_size = config['batch_size']
    opt_name = config['optimizer']
    lr = config['lr']
    epochs = config['epochs']

    print(f"\n--- Running: {ds_name} | {model_name} | {opt_name} | Batch {batch_size} | LR {lr} ---")

    # Get Data
    train_loader, val_loader, test_loader = get_dataloaders(ds_name, batch_size)

    # Get Model
    model = get_model(model_name)

    # Optimizer
    if opt_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)

    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler() # For Automatic Mixed Precision (AMP)

    # Training Loop
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # AMP Forward Pass
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            # AMP Backward Pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        # Validation Loop (Optional: Add print if you want to track progress)
        # We skip printing every epoch to save space, but you can add it back.

    # FINAL TEST EVALUATION
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Result >> Test Accuracy: {acc:.2f}%")
    return acc

# --- 4. EXPERIMENT RUNNER ---
# This matches the table in your assignment image
# You can add FashionMNIST to this list to do Q1(a) fully.
# --- 4. EXPERIMENT RUNNER (MODIFIED FOR BOTH DATASETS) ---
experiments = [
    # Format: (Batch, Optim, LR, Model)
    (16, 'SGD', 0.001, 'resnet18'),
    (16, 'SGD', 0.0001, 'resnet18'),
    (16, 'Adam', 0.001, 'resnet18'),
    (16, 'Adam', 0.0001, 'resnet18'),

    (16, 'SGD', 0.001, 'resnet50'),
    (16, 'SGD', 0.0001, 'resnet50'),
    (16, 'Adam', 0.001, 'resnet50'),
    (16, 'Adam', 0.0001, 'resnet50'),

    (32, 'SGD', 0.001, 'resnet18'),
    (32, 'Adam', 0.0001, 'resnet18'),

    (32, 'SGD', 0.001, 'resnet50'),
    (32, 'Adam', 0.0001, 'resnet50'),

    # NOTE: You can add the Batch 32 experiments here if you need them later
]

datasets_to_run = ['MNIST', 'FashionMNIST']

print(f"Total Configurations to run: {len(datasets_to_run) * len(experiments)}")

for ds_name in datasets_to_run:
    print(f"\n{'='*20} STARTING {ds_name} EXPERIMENTS {'='*20}")

    for exp in experiments:
        batch, opt, lr, model = exp
        config = {
            'dataset': ds_name,   # Loops through MNIST then FashionMNIST
            'batch_size': batch,
            'optimizer': opt,
            'lr': lr,
            'model': model,
            'epochs': 5  # Keep this low (e.g. 5) to finish before the deadline!
        }

        # Run the training
        train_and_evaluate(config)

print("\nALL EXPERIMENTS COMPLETED.")